# HAI Meta-Analysis

The following analysis is modeled off the template here: https://osf.io/5nk92

Moreau, D., & Gamble, B. (2022). Conducting a Meta-Analysis in the Age of Open Science: Tools, Tips, and Practical Recommendations. Psychological Methods, 27 (3), 426-432.

## Set-Up

In [ ]:
# Install packages using the pacman package
if (!require("pacman")) install.packages("pacman")
pacman::p_load(tidyverse, metafor, dplyr, ggplot2, readr, gtsummary, broom, rlang, forestploter)

# Import data
df <- read.csv("Data_Extraction_communication.csv", fileEncoding="UTF-8-BOM", na.strings = "NA")

## Descriptive Statistics, Table

In [ ]:
# Number of unique publications, experiments, effect sizes
n_pub <- length(unique(df$Paper_ID)) # Number of unique publications
n_exp <- length(unique(df$Exp_ID_Cleaned)) # Number of unique experiments
n_es <- length(unique(df$ES_ID)) # Number of unique effect sizes

print("n_pub")
print(n_pub)
print("n_exp")
print(n_exp)
print("n_es")
print(n_es)

# Summary statistics of ES characteristics
df %>% 
  select(Baseline, Comp_Type, AI_Expl_Incl, AI_Conf_Incl, 
         Task_Type, Marketing_Task, Task_Data_Cleaned,
         # Task_Data_IsCategoric, Task_Data_IsCode, Task_Data_IsImage, Task_Data_IsNumeric, Task_Data_IsText, Task_Data_IsVideo, Task_Output, 
         Task_Output_Cleaned, 
         AI_Type_Cleaned, 
         Participant_Expert,
         Participant_Crowdworker,
         Year,
         Perf_Metric_Cleaned) %>%  
  tbl_summary() %>% 
  modify_header(label ~ "**Effect Sizes**") %>%
  modify_caption("**Table X. Descriptive Statistics**")

## Descriptive Statistics, Bar Plots

In [ ]:
# Define the variable that contains the column name
col_names <- c("Baseline_Cleaned", "Task_Type", "Marketing_Task", "Task_Output_Cleaned", "Division_Labor",
               "AI_Type_Cleaned", "AI_Expl_Incl", "AI_Conf_Incl",
               "Comp_Type_Cleaned", "Participant_Crowdworker", "Participant_Expert", "Perf_Metric_Cleaned")
titles <- c("Do Humans or AI\nPerform Better Alone?", 
            "Type of Task",
            "Marketing-related Tasks",
            "Task Ouput", 
            "Pre-Determined\nDivision of Labor",
            "Type of AI",
            "AI Explanation Included",
            "AI Confidence Included",
            "Experimental Design",
            "Crowdworker Participants",
            "Expert Participants",
            "Task Performance")

col <- "#0072B2"
plot_list <- list()

df$Baseline_Cleaned <- ifelse(df$Baseline == "Human", "Humans", "AI")
df$Comp_Type_Cleaned <- ifelse(df$Comp_Type == "Independent Samples", "Within\nSubjects", "Between\nSubjects")

df_plot <- df %>% mutate(Task_Output_Cleaned = ifelse(Task_Output_Cleaned == "Open Response", "Open", Task_Output_Cleaned))

for(i in 1:length(col_names)) {
  col_name <- col_names[i]
  title <- titles[i]
  fig_name <- paste("Figures/barplot_", col_name, ".png", sep = "")
  
  if(i %% 3 == 1) {
    yaxt = "s"
    ylab = "Number of Effect Sizes"
  }
  if(i %% 3 != 1) {
    yaxt = "s"
    ylab = ""
  }
  
  # Get counts for each category
  counts <- table(df[,col_name])
  
  # Make ymax a multiple of 5
  ymax <- ceiling(max(counts + 25) / 25) * 25
  ymax <- 170
  
  # Open figure
  png(fig_name, width = 700, height = 700, units = "px")
  par(mar = c(12, 8, 6, 4))
  # Define the barplot and save midpoints
  labels <- levels(as.factor(df_plot[,col_name]))
  # give more room for diagonal labels
  par(mar = c(14, 8, 6, 4))

  # 1) draw bars without names
  midpoints <- barplot(
    counts,
    names.arg = rep("", length(labels)),    # no built-in labels
    las       = 1,                          # leave them horizontal for now
    ylab      = ylab,
    yaxt      = yaxt,
    main      = title,
    col       = col,
    border    = NA,
    ylim      = c(0, ymax),
    cex.axis  = 2,
    cex.lab   = 1.5,
    cex.main  = 3
  )
  
  # 2) compute a little y-offset below the axis
  y_off <- par("usr")[3] - diff(par("usr")[3:4]) * 0.05
  
  # 3) add your own smaller, 45° labels
  text(
    x      = midpoints,
    y      = y_off,
    labels = labels,
    srt    = 45,      # rotate
    adj    = c(1,1),  # right/top justify
    xpd    = TRUE,    # allow into margin
    cex    = 1.7      # label size
  )
    
  # Calculate percentage for each bar
  perc <- round(counts / sum(counts) * 100)
  
  # Add x-axis
  # abline(h = 0, col = "black", lwd = 2)
  
  # Add counts and percentages on top of each bar
  text(x = midpoints, y = counts, 
       labels = paste(counts, " (", perc, "%)", sep = ""), 
       pos = 3, col = "black", cex = 2.3)
  
  # Save the plot to a variable
  plot <- recordPlot()
  plot_list <- append(plot_list, plot)
  plot
  
  dev.off()
}

## Descriptive Statistics (Histograms)

In [ ]:
col_name <- "N_Exp"
fig_name <- paste("Figures/barplot_", col_name, ".png", sep = "")
png(fig_name, width = 1000, height = 1000, units = "px")
par(mar = c(12, 8, 6, 4))
xlab <- "Number of Participants"
ylab <- "Number of Effect Sizes"
title <- "Number of Participants"

p <- hist(df$N_Exp,
     xlab = xlab,
     ylab = ylab,
     main = title, 
     col = col, 
     border = NA, 
     cex.axis = 2,
     cex.main = 3,
     cex.lab = 2)

p

dev.off()

## Calculate Effect Sizes

In [ ]:
# See https://wviechtb.github.io/metafor/reference/escalc.html for formulas

# Strong synergy (HAI > max(H, AI))
es_from_mean_sd_s <- escalc(measure = "SMD", vtype="LS", m1i = Avg_Perf_HumanAI_Adj,           
                            m2i = Avg_Perf_Baseline_Adj, sd1i = Sd_Perf_HumanAI,
                            sd2i = Sd_Perf_Baseline, n1i = N_HumanAI, n2i = N_Human,
                            data = df, slab = ES_ID)            

# Weak synergy (HAI > H)
es_from_mean_sd_h <- escalc(measure = "SMD", vtype="LS", m1i = Avg_Perf_HumanAI_Adj,           
                            m2i = Avg_Perf_Human_Adj, sd1i = Sd_Perf_HumanAI,
                            sd2i = Sd_Perf_Human, n1i = N_HumanAI, n2i = N_HumanAI,
                            data = df, slab = ES_ID) 

# Weak synergy (HAI > AI)
es_from_mean_sd_a <- escalc(measure = "SMD", vtype="LS", m1i = Avg_Perf_HumanAI_Adj,           
                            m2i = Avg_Perf_AI_Adj, sd1i = Sd_Perf_HumanAI,
                            sd2i = Sd_Perf_AI, n1i = N_HumanAI, n2i = N_Human,
                            data = df, slab = ES_ID) 

# Negative synergy (HAI < min(H, AI))
es_from_mean_sd_n <- escalc(measure = "SMD", vtype="LS", m1i = Avg_Perf_HumanAI_Adj,           
                            m2i = Avg_Perf_Worse_Adj, sd1i = Sd_Perf_HumanAI,
                            sd2i = Sd_Perf_Worse, n1i = N_HumanAI, n2i = N_Human,
                            data = df, slab = ES_ID) 

es_s <- es_from_mean_sd_s$yi
variance_s <- es_from_mean_sd_s$vi
es_h <- es_from_mean_sd_h$yi
variance_h <- es_from_mean_sd_h$vi
es_a <- es_from_mean_sd_a$yi
variance_a <- es_from_mean_sd_a$vi
es_n <- es_from_mean_sd_n$yi
variance_n <- es_from_mean_sd_n$vi

# Append the new effect sizes and variances to the main data frame
df <- cbind(df, es_s, variance_s)
df <- cbind(df, es_h, variance_h)
df <- cbind(df, es_a, variance_a)
df <- cbind(df, es_n, variance_n)

# col_select <- c("ES_ID","es_s", "variance_s","es_h", "variance_h","es_a", "variance_a")
# df_new <- data.frame(ES_ID = df$ES_ID, Exp_ID_Cleaned = df$Exp_ID_Cleaned, es_h, variance_h, # es_s, variance_s, es_n, variance_n, es_a, variance_a)

In [ ]:
# Define the variables for effect size
# Strong synergy (HAI > max(H, AI))
m1i<-df$Avg_Perf_HumanAI_Adj
m2i<-df$Avg_Perf_Baseline_Adj
sd1i<-df$Sd_Perf_HumanAI
sd2i<-df$Sd_Perf_Baseline
n1i<- df$N_HumanAI
n2i<- df$N_Human


# # Weak synergy (HAI > H)
# m1i<-df$Avg_Perf_HumanAI_Adj
# m2i<-df$Avg_Perf_Human_Adj
# sd1i<-df$Sd_Perf_HumanAI
# sd2i<-df$Sd_Perf_Human
# n1i<-df$N_HumanAI
# n2i<-df$N_Human
# 
# # Weak synergy (HAI > AI)
# m1i<-df$Avg_Perf_HumanAI_Adj
# m2i<-df$Avg_Perf_AI_Adj
# sd1i<-df$Sd_Perf_HumanAI
# sd2i<-df$Sd_Perf_AI
# n1i<-df$N_HumanAI
# n2i<-df$N_Human
                     
# 1) Compute raw percent difference for each effect size
#    pdiff = (mean_HAI - mean_Baseline) / mean_Baseline * 100
df$pdiff <- (m1i - m2i) / abs(m2i) * 100

# 2) Approximate its variance via the delta method (so you could meta-analyze pdiff too)
#    var(pdiff) ≈ (1 / m2i^2) * var(m1i) + (m1i^2 / m2i^4) * var(m2i)
#    where var(mi) = sd_i^2 / n_i
df$var_pdiff <- (1 /m2i^2) * (sd1i^2 / n1i) +
                 (m1i^2 /m2i^4) * (sd2i^2 /n2i)

# 3) Summarize percent differences across studies
library(dplyr)
summary_pdiff <- df %>%
  summarize(
    mean_pdiff   = mean(pdiff,   na.rm = TRUE),
    sd_pdiff     = sd(pdiff,     na.rm = TRUE),
    k            = sum(!is.na(pdiff)),
    se_pdiff     = sd_pdiff / sqrt(k),
    ci_lower     = mean_pdiff - qt(0.975, df = k - 1) * se_pdiff,
    ci_upper     = mean_pdiff + qt(0.975, df = k - 1) * se_pdiff
  )

# 4) Print the summary table
print(summary_pdiff)

# 5) (Optional) Run a random-effects meta-analysis on pdiff itself
library(metafor)
rma_pdiff <- rma(yi = pdiff, vi = var_pdiff, data = df, method = "REML")
summary(rma_pdiff)

## 4. Fit the main meta-analytic model

In [ ]:
# 4.1 Fit the multi-level meta-analytic model with no moderators
# Perform cluster robust inference, cluster = experiment
synergy_types <- c("Strong", "Human", "AI", "Negative")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Exp_ID_Cleaned/ES_ID)   
  main_model_robust <- robust(main_model, cluster = Exp_ID_Cleaned, clubSandwich=TRUE)
  print(summary(main_model_robust))
}

## Forest Plots

In [ ]:
synergy_types <- c( "AI", "Strong", "Human")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
    
    fig_name = paste("Figures/ForestPlot_Synergy_", "S.png", sep = "")
    title <- "Strong Synergy"
    subtitle <- expression(italic("Human-AI System versus max(Human, AI)"))
    neg_label_part <- "The human-AI group\nunderperforms either the\nhuman or AI alone\n(n = "
    pos_label_part <- "The human-AI group\noutperforms both the\nhuman and AI alone\n(n = "
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
    
    fig_name = paste("Figures/ForestPlot_Synergy_", "H.png", sep = "")
    title <- "Weak Synergy"
    subtitle <- expression(italic("Human-AI System versus Human Alone"))
    neg_label_part <- "The human-AI group\nunderperforms the\nhuman alone\n(n = "
    pos_label_part <- "The human-AI group\noutperforms the\nhuman alone\n(n = "
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
    
    fig_name = paste("Figures/ForestPlot_Synergy_", "A.png", sep = "")
    title <- "Weak Synergy"
    subtitle <- expression(italic("Human-AI System versus AI Alone"))
    neg_label_part <- "The human-AI group\nunderperforms the\nAI alone\n(n = "
    pos_label_part <- "The human-AI group\noutperforms the\nAI alone\n(n = "
  }
}

main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Exp_ID_Cleaned/ES_ID)   
main_model_robust <- robust(main_model, cluster = Exp_ID_Cleaned, clubSandwich=TRUE)

k <- main_model_robust$k
n_pos_es <- sum(df$es > 0)
n_neg_es <- sum(df$es <= 0)
es_color <- ifelse(df$es > 0, "darkgreen", "darkred")

neg_label <- paste(neg_label_part, n_neg_es, ", ", round(n_neg_es/k * 100,1), "%)", sep = "")
neg_label_pos <- min((n_pos_es + k) / 2, k - 22)
pos_label <- paste(pos_label_part, n_pos_es, ", ", round(n_pos_es/k * 100,1), "%)", sep = "")
pos_label_pos <- n_pos_es / 2

xlim <- 6
estimate <- coef(main_model_robust)
width <- 800*res/72
height <- 1200*res/72
res <- 300
cex <- 1.5
sum_pos <- -8
title_pos <- main_model_robust$k + 23
subtitle_pos <- main_model_robust$k + 10

est_color <- ifelse(estimate > 0, "darkgreen", "darkred")
caption <- paste("Model Estimate: g = ", round(estimate, 2), " [", round(main_model_robust$ci.lb, 2), 
                 ",", round(main_model_robust$ci.ub, 2), "]", sep = "")

png(fig_name, width = width, height = height, res = res)

par(mar = c(4, 5, 2, 3))
metafor::forest(main_model_robust,
       xlim=c(-1*xlim,xlim),        ### adjust horizontal plot region limits
       # ylim=c(-2, main_model_robust$k+15),       # add space at top of plot
       alim=c(-1*xlim,xlim),        ### adjust x-axis limits
       order="obs",             ### order by size of yi
       slab=NA, annotate=FALSE, ### remove study labels and annotations
       efac=0,                  ### remove vertical bars at end of CIs
       pch=19,                  ### changing point symbol to filled circle
       psize=2,                 ### increase point size
       cex.lab=cex, cex.axis=cex,   ### increase size of x-axis title/labels
       lty=c("solid","solid"),
       addfit = FALSE,
       colout = es_color, # give a vector of scaled colors red to green
       # colout = colors,
       xlab = "Size of the Effects (Hedges' g) with 95% Confidence Intervals",
       top = 15 # specify the amount of space to leave empty at the top of the plot
       )  ### remove horizontal line at top of plot
 
abline(h=0) # vertical line at zero
polygon(c(-1*xlim, 0, 0, -1*xlim), c(-15, -15, main_model_robust$k+1, main_model_robust$k+1), 
        col = rgb(139/255, 0, 0, alpha = 0.2), border = NA) # shading1
polygon(c(xlim, 0, 0, xlim), c(-15, -15, main_model_robust$k+1, main_model_robust$k+1), 
        col = rgb(0, 100/255, 0, alpha = 0.2), border = NA) # shading2
polygon(c(-1*xlim, 0, 0, -1*xlim), c(n_pos_es, n_pos_es, main_model_robust$k+1, main_model_robust$k+1),
        col = rgb(139/255, 0, 0, alpha = 0.3), border = NA) # shading dark
polygon(c(xlim, 0, 0, xlim), c(-15, -15, main_model_robust$k+1 - n_neg_es, main_model_robust$k+1 - n_neg_es),
        col = rgb(0, 100/255, 0, alpha = 0.3), border = NA) # shading dark

points(estimate, sum_pos, pch=19, col = est_color, cex = 0.75) # summary estimate
segments(x0 = main_model_robust$ci.lb, x1 = main_model_robust$ci.ub,
         y0 = sum_pos, y1 = sum_pos, lwd = 2, col = est_color) # summary CI
if(estimate < 0) {
  text(-1*xlim, sum_pos, caption, pos=4, offset=0, cex=0.75) # label summary
}
if(estimate >= 0) {
  text(xlim, sum_pos, caption, pos=2, offset=0, cex=0.75) # label summary
}

text(0, title_pos, title, cex = 3) # title
text(0, subtitle_pos, subtitle, cex = 2) # title
text(xlim, pos_label_pos, pos_label, pos=2, offset=0, cex=1.75) # caption1
text(-xlim, neg_label_pos, neg_label, pos=4, offset=0, cex=1.75) # caption2

dev.off()

## Heterogeneity analysis

In [ ]:
synergy_types <- c("Strong", "Human", "AI", "Negative")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Exp_ID_Cleaned/ES_ID)   
  main_model_robust <- robust(main_model, cluster = Exp_ID_Cleaned, clubSandwich=TRUE)

  # Tau-squared statistic (total heterogeneity)
  tau_squared <- sum(main_model_robust$sigma2)
  print(tau_squared)
  
  # I-Squared statistic (ratio of true to total variance)
  # Using the formula from www.metafor-project.org/doku.php/tips:i2_multilevel_multivariate
  W <- diag(1/df$variance)
  X <- model.matrix(main_model_robust)
  P <- W - W %*% X %*% solve(t(X) %*% W %*% X) %*% t(X) %*% W
  I2_statistic <- 100 * sum(main_model_robust$sigma2) / 
    (sum(main_model_robust$sigma2) + 
       (main_model_robust$k-main_model_robust$p)/sum(diag(P)))
  print(I2_statistic)
  
  # Percentage of I-Squared from within-experiments versus between-experiments
  # Using the formula from www.metafor-project.org/doku.php/tips:i2_multilevel_multivariate
  I2_pct <- 100 * main_model_robust$sigma2 / (sum(main_model_robust$sigma2) + (main_model_robust$k-main_model_robust$p)/sum(diag(P)))
  print(I2_pct) # between-experiments (level 3), within-experiments (level 2)
}

## Moderator Analysis (No Intercept)

In [ ]:
synergy_types <- c("Strong", "Human", "AI", "Negative")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
    file_name = paste("Data/ModeratorAnalysis_Synergy_", "S.csv", sep = "")
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
    file_name = paste("Data/ModeratorAnalysis_Synergy_", "H.csv", sep = "")
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
    file_name = paste("Data/ModeratorAnalysis_Synergy_", "A.csv", sep = "")
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
    file_name = paste("Data/ModeratorAnalysis_Synergy_", "N.csv", sep = "")
  }
  
  mods <- c(#"Baseline",
            "Task_Type",
            "Marketing_Task",
            "Task_Output_Cleaned",
            "Task_Data_Cleaned",
            "Year",
            "AI_Type_Cleaned",
            "AI_Expl_Incl",
            "AI_Conf_Incl",
            "Participant_Expert",
            "Participant_Crowdworker",
            "Comp_Type",
            "Division_Labor"
            )

  df_all_results <- data.frame()
  
  for(mod in mods) {
    print(mod)
    k_es <- c()
    vals <- sort(unique(df[[mod]]))
  
    for(val in vals) {
      k_es_val <- sum(df[[mod]] == val)
      k_es <- append(k_es, k_es_val)
    }
    
    model_formula <- as.formula(paste("~", quo_name(mod), "- 1"))
    
    if(mod == "Year") {
      model_formula <- as.formula(paste("~as.factor(", quo_name(mod), ")- 1"))
    }
    
    model_mod <- rma.mv(es, variance, data = df,
                       mods = model_formula, 
                       tdist = TRUE,
                       method = "REML", level = 95, random = ~ 1 | Exp_ID_Cleaned/ES_ID)
    model_mod_robust <- robust(model_mod, cluster = Exp_ID_Cleaned, adjust=TRUE)
    
    df_result <- tidy(model_mod_robust, conf.int = TRUE)
    df_result$Moderator <- mod
    df_result$Value <- vals
    df_result$k_es <- k_es
    #df_result$k_exp <- k_exp
    
    df_all_results <- rbind(df_all_results, df_result)
    print(summary(model_mod_robust))
  }
  write.csv(df_all_results, file = file_name, row.names = FALSE)
}

## Moderator Analysis (with intercept)

In [ ]:
synergy_types <- c("Strong", "Human", "AI")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
      file_name = paste("Data/ModeratorAnalysis_Synergy_Intercept_", "S.csv", sep = "")
      df$es <- df$es_s
      df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
      file_name = paste("Data/ModeratorAnalysis_Synergy_Intercept_", "H.csv", sep = "")
      df$es <- df$es_h
      df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
      file_name = paste("Data/ModeratorAnalysis_Synergy_Intercept_", "A.csv", sep = "")
      df$es <- df$es_h
      df$variance <- df$variance_h
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
    file_name = paste("Data/ModeratorAnalysis_Synergy_Intercept", "N.csv", sep = "")
  }
  
  mods <- c(#"Baseline",
            "Task_Type",
            "Task_Output_Cleaned",
            "Task_Data_Cleaned",
            "Year",
            "AI_Type_Cleaned",
            "AI_Expl_Incl",
            "AI_Conf_Incl",
            "Participant_Expert",
            "Participant_Crowdworker",
            "Comp_Type",
            "Division_Labor"
            )

  df_all_results <- data.frame()
  
  for(mod in mods) {
    print(mod)
    k_es <- c()
    vals <- sort(unique(df[[mod]]))
  
    for(val in vals) {
      k_es_val <- sum(df[[mod]] == val)
      k_es <- append(k_es, k_es_val)
    }
    
    model_formula <- as.formula(paste("~", quo_name(mod)))
    
    if(mod == "Year") {
      model_formula <- as.formula(paste("~as.factor(", quo_name(mod), ")"))
    }
    
    model_mod <- rma.mv(es, variance, data = df,
                       mods = model_formula, 
                       tdist = TRUE,
                       method = "REML", level = 95, random = ~ 1 | Exp_ID_Cleaned/ES_ID)
    model_mod_robust <- robust(model_mod, cluster = Exp_ID_Cleaned, adjust=TRUE)
    df_result <- tidy(model_mod_robust, conf.int = TRUE)
    df_result$Moderator <- mod
    df_result$Value <- vals
    df_result$k_es <- k_es
    #df_result$k_exp <- k_exp
    
    df_all_results <- rbind(df_all_results, df_result)
    print(summary(model_mod_robust))
  }
  write.csv(df_all_results, file = file_name, row.names = FALSE)
}

## Funnel Plots

In [ ]:
synergy_types <- c("Human", "AI", "Strong", "Negative")

width <- 1600
height <- 1400

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
    title <- "Strong Synergy: Human-AI System versus max(Human, AI)"
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
    title <- "Weak Synergy: Human-AI System versus Human Alone"
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
    title <- "Weak Synergy: Human-AI System versus AI Alone"
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
    title <- "Negative Synergy: Human-AI System versus min(Human, AI)"
  }
  
  fig_name = paste("Figures/FunnelPlot_Synergy_", synergy_type, ".png", sep = "")
  main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Exp_ID_Cleaned/ES_ID)   
  main_model_robust <- robust(main_model, cluster = Exp_ID_Cleaned, clubSandwich=TRUE)
  
  estimate <- coef(main_model_robust)
  
  png(fig_name, width = width, height = height, res = res)

  funnel(main_model_robust,                    
         # xlim=c(-6, 6), 
         xlab= sprintf("Size of the Effects (Hedge's g)\n%s", title), 
         ylab= "Standard Error", 
         ylim = c(1.2, 0), 
         steps = 6, 
         digits = c(1, 2),
         level=c(95), 
         shade=c("white", "gray75"), 
         col = "#0072B2",
         refline=estimate, 
         # legend=TRUE,
         # pch=ifelse(yi > -1, 19, 21), 
         # col=ifelse(sqrt(vi) > .3, "red", "blue"),
         title = title,
         subtitle = subtitle, 
         label="Exp_ID"
         )
  
  dev.off()
}

## Bias Tests

In [ ]:
synergy_types <- c("Human", "AI", "Strong", "Negative")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  # Egger's test
  eggers_model <- metafor::rma.mv(es, variance, data = df, mod = ~ sqrt(variance), 
                                    tdist = TRUE, method = "REML", level = 95,              
                                    digits = 7, slab = ES_ID, random = ~ 1 | Exp_ID_Cleaned/ES_ID)
  eggers_model_robust <- robust(eggers_model, cluster = Exp_ID_Cleaned, adjust = TRUE)
  summary <- summary(eggers_model_robust)
  print(summary)
  
  # Rank test
  main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Exp_ID_Cleaned/ES_ID)   
  main_model_robust <- robust(main_model, cluster = Exp_ID_Cleaned, clubSandwich=TRUE)
  ranktest <- ranktest(main_model_robust)
  print(ranktest)
}

## Sensitivity analysis: Fit the main meta-analytic model, clustering at the paper level

In [ ]:
synergy_types <- c("Human", "AI", "Strong", "Negative")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Paper_ID/ES_ID)   
  main_model_robust <- robust(main_model, cluster = Paper_ID, clubSandwich=TRUE)
  print(summary(main_model_robust))
}

## Sensitivity analysis: Calculate outliers and influential points

In [ ]:
synergy_types <- c("Strong", "Human", "AI")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  main_model <- rma.mv(yi = es, V = variance, data = df, tdist=TRUE, method = "REML",         
                     level = 95, digits = 7, slab = ES_ID, random = ~ 1 | Paper_ID/ES_ID)   
  main_model_robust <- robust(main_model, cluster = Paper_ID, clubSandwich=TRUE)
  
  # Standardized residuals of correlations (each effect size)
  resid_es <- rstandard(main_model)
  outliers_resid_es <- resid_es %>%
    subset(resid_es$z > 3 | resid_es$z < -3) # determine outliers
  print(outliers_resid_es)
  
  # Cooks distance (each effect size)
  cooks_es <- cooks.distance(main_model_robust, progbar=TRUE) # takes a long time to run
  threshold <- 4 / (length(df$ES_ID))
  outliers_cooks_es <- cooks_es %>%
    subset(cooks_es > threshold) # determine influence
  print(outliers_cooks_es)
  
  df_sens <- cbind(df, resid = resid_es$z, cooks = cooks_es)

  # Perform re-analysis on dataset excluding outliers and influential points
  df_sens <- filter(df_sens, (abs(df_sens$resid) < 3) & (df_sens$cooks < threshold)) 
  main_model_sens <- rma.mv(yi = es, 
                  V = variance,
                  data = df_sens,
                  method = "REML",         # restricted maximum likelihood
                  level = 95,              # CI
                  digits = 7,              # decimal points
                  slab = ES_ID,          # paper labels
                  random = ~ 1 | Exp_ID_Cleaned/ES_ID # multilevel model to handle sample dependency
                  )  
  print(summary(main_model_sens))
  
  main_model_sens_robust <- robust(main_model_sens, cluster = df_sens$Exp_ID_Cleaned)
  print(summary(main_model_sens_robust))
  
 # Outliers
  file_name <- paste("Data/Outliers_", synergy_type, ".csv", sep = "")
  df_outliers <- df_sens %>% filter(abs(resid) > 3 | cooks > threshold)

  write.csv(df_outliers, file_name, row.names = FALSE)
}

## Sensitivity Analysis: Leave-one-out, effect size

In [ ]:
synergy_types <- c("Strong", "Human")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  # Leave-one-out analysis at effect size level.
  df_loo_es <- matrix(0, ncol = 3, nrow = length(unique(df$ES_ID)))
  df_loo_es <- data.frame(df_loo_es)
  colnames(df_loo_es) <- c("est", "p_val", "I2_statistic")
  i = 1
  for(es_id in unique(df$ES_ID))
  {
    df_loo_sub <- subset(df, ES_ID != es_id)
    
    loo_model <- rma.mv(yi = es, 
                  V = variance,
                  data = df_loo_sub,
                  method = "REML",         # restricted maximum likelihood
                  level = 95,              # CI
                  digits = 7,              # decimal points
                  slab = ES_ID,          # paper labels
                  random = ~ 1 | Exp_ID_Cleaned/ES_ID # multilevel model to handle sample dependency
                  )
    
    loo_model_robust <- robust(loo_model, cluster = df_loo_sub$Exp_ID_Cleaned, adjust = TRUE)
    
    W <- diag(1/df_loo_sub$variance)
    X <- model.matrix(loo_model_robust)
    P <- W - W %*% X %*% solve(t(X) %*% W %*% X) %*% t(X) %*% W
    I2_statistic <- 100 * sum(loo_model_robust$sigma2) / 
      (sum(loo_model_robust$sigma2) + 
         (loo_model_robust$k-loo_model_robust$p)/sum(diag(P)))
    
    df_loo_es[i, 1] <- loo_model_robust$b
    df_loo_es[i, 2] <- loo_model_robust$pval
    df_loo_es[i, 3] <- I2_statistic
    i <- i + 1
  }
  print(range(df_loo_es$est))
  print(range(df_loo_es$p_val))
  print(range(df_loo_es$I2_statistic))
}

## Sensitivity Analysis: Leave-one-out, experiment

In [ ]:
synergy_types <- c("Human", "AI", "Strong")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  # Leave-one-out analysis at experiment level.
  df_loo_exp <- matrix(0, ncol = 3, nrow = length(unique(df$Exp_ID_Cleaned)))
  df_loo_exp <- data.frame(df_loo_exp)
  colnames(df_loo_exp) <- c("est", "p_val", "I2_statistic")
  i = 1
  for(exp_id in unique(df$Exp_ID_Cleaned))
  {
    df_loo_sub <- subset(df, Exp_ID_Cleaned != exp_id)
    
    loo_model <- rma.mv(yi = df_loo_sub$es, 
                  V = df_loo_sub$variance,
                  data = df_loo_sub,
                  method = "REML",         # restricted maximum likelihood
                  level = 95,              # CI
                  digits = 7,              # decimal points
                  slab = ES_ID,          # paper labels
                  random = ~ 1 | Exp_ID_Cleaned/ES_ID # multilevel model to handle sample dependency
                  )
    
    loo_model_robust <- robust(loo_model, cluster = df_loo_sub$Exp_ID_Cleaned, adjust = TRUE)
    
    W <- diag(1/df_loo_sub$variance)
    X <- model.matrix(loo_model_robust)
    P <- W - W %*% X %*% solve(t(X) %*% W %*% X) %*% t(X) %*% W
    I2_statistic <- 100 * sum(loo_model_robust$sigma2) / 
      (sum(loo_model_robust$sigma2) + 
         (loo_model_robust$k-loo_model_robust$p)/sum(diag(P)))
    
    df_loo_exp[i, 1] <- loo_model_robust$b
    df_loo_exp[i, 2] <- loo_model_robust$pval
    df_loo_exp[i, 3] <- I2_statistic
    i <- i + 1
  }
  print(range(df_loo_exp$est))
  print(range(df_loo_exp$p_val))
  print(range(df_loo_exp$I2_statistic))
}

## Sensitivity Analysis: Leave-one-out, paper

In [ ]:
synergy_types <- c("Human", "AI", "Strong")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
    df$es <- df$es_s
    df$variance <- df$variance_s
  }
  if(synergy_type == "Human") {
    df$es <- df$es_h
    df$variance <- df$variance_h
  }
  if(synergy_type == "AI") {
    df$es <- df$es_a
    df$variance <- df$variance_a
  }
  if(synergy_type == "Negative") {
    df$es <- df$es_n
    df$variance <- df$variance_n
  }
  
  # Leave-one-out analysis at paper level.
  df_loo_pap <- matrix(0, ncol = 3, nrow = length(unique(df$Paper_ID)))
  df_loo_pap <- data.frame(df_loo_pap)
  colnames(df_loo_pap) <- c("est", "p_val", "I2_statistic")
  i = 1
  for(pap_id in unique(df$Paper_ID))
  {
    df_loo_sub <- subset(df, Paper_ID != pap_id)
    
    loo_model <- rma.mv(yi = df_loo_sub$es, 
                  V = df_loo_sub$variance,
                  data = df_loo_sub,
                  method = "REML",         # restricted maximum likelihood
                  level = 95,              # CI
                  digits = 7,              # decimal points
                  slab = ES_ID,          # paper labels
                  random = ~ 1 | Exp_ID_Cleaned/ES_ID # multilevel model to handle sample dependency
                  )
    
    loo_model_robust <- robust(loo_model, cluster = df_loo_sub$Exp_ID_Cleaned, adjust = TRUE)
    
    W <- diag(1/df_loo_sub$variance)
    X <- model.matrix(loo_model_robust)
    P <- W - W %*% X %*% solve(t(X) %*% W %*% X) %*% t(X) %*% W
    I2_statistic <- 100 * sum(loo_model_robust$sigma2) / 
      (sum(loo_model_robust$sigma2) + 
         (loo_model_robust$k-loo_model_robust$p)/sum(diag(P)))
    
    df_loo_pap[i, 1] <- loo_model_robust$b
    df_loo_pap[i, 2] <- loo_model_robust$pval
    df_loo_pap[i, 3] <- I2_statistic
    i <- i + 1
  }
  print(range(df_loo_pap$est))
  print(range(df_loo_pap$p_val))
  print(range(df_loo_pap$I2_statistic))
}

## How does the performance of the human and AI alone impact the potential for different types of synergy?

In [ ]:
synergy_types <- c("Strong", "Human", "AI", "Negative")

for(synergy_type in synergy_types) {
  if(synergy_type == "Strong") {
      df$es <- df$es_s
      fig_name = paste("Figures/Acc_Synergy_", "S.png", sep = "")
      title <- "Strong Synergy in Decision Tasks"
      subtitle <- expression(italic("Human-AI System versus max(Human, AI)"))
  }
  if(synergy_type == "Human") {
      df$es <- df$es_h
      fig_name = paste("Figures/Acc_Synergy_", "H.png", sep = "")
      title <- "Weak Synergy in Decision Tasks"
      subtitle <- expression(italic("Human-AI System versus Human Alone"))
  } 
  if(synergy_type == "AI") {
      df$es <- df$es_a
      fig_name = paste("Figures/Acc_Synergy_", "A.png", sep = "")
      title <- "Weak Synergy in Decision Tasks"
      subtitle <- expression(italic("Human-AI System versus AI Alone"))
  } 
  if(synergy_type == "Negative") {
      df$es <- df$es_n
      fig_name = paste("Figures/Acc_Synergy_", "N.png", sep = "")
      title <- "Negative Synergy in Decision Tasks"
      subtitle <- expression(italic("Human-AI System versus min(Human, AI)"))
  }
  
  df_acc <- filter(df, Perf_Metric_Cleaned == "Accuracy")
  n_pos_es <- sum(df_acc$es > 0)
  n_neg_es <- sum(df_acc$es <= 0)
  df_acc$Avg_Perf_Human_Adj_Cleaned <- df_acc$Avg_Perf_Human_Adj * 100
  df_acc$Avg_Perf_AI_Adj_Cleaned <- df_acc$Avg_Perf_AI_Adj * 100
  
  point_size <- 2
  
  p <- ggplot(df_acc, aes(x = Avg_Perf_Human_Adj_Cleaned, y = Avg_Perf_AI_Adj_Cleaned, color = ifelse(es > 0, "Positive", "Negative"))) +
    geom_point(size = point_size) +
    scale_color_manual(values = c("Positive" = "darkgreen", "Negative" = "darkred"),
                       labels = c(sprintf("No (n = %d, %.1f%%)", n_neg_es, n_neg_es / (n_neg_es + n_pos_es) * 100), 
                                  sprintf("Yes (n = %d, %.1f%%)", n_pos_es, n_pos_es / (n_neg_es + n_pos_es) * 100)),
                       name = "Human-AI Synergy") +
    theme_linedraw() + 
    geom_abline(intercept = 0, slope = 1, linetype = "dashed", color = "gray") +
    labs(title = title, 
         subtitle = subtitle,
         x = "Accuracy of Human Alone (%)", 
         y = "Accuracy of AI Alone (%)") +
    # theme(legend.position = "none") 
    theme(# legend.position = c(0, 1), 
          legend.justification = c(0, 1),
          legend.box.just = "left",
          legend.key = element_rect(fill = "transparent"),
          legend.background = element_rect(fill = "transparent"),
          # panel.grid.minor = element_blank(),
          plot.title = element_text(size = 16, face = "bold", hjust = 0.5),
          plot.subtitle = element_text(size = 14, hjust = 0.5),
          legend.title = element_text(size = 14),
          axis.text.x = element_text(size = 12),
          axis.text.y = element_text(size = 12),
          axis.title.x = element_text(size = 14),
          axis.title.y = element_text(size = 14),
          legend.text = element_text(size = 12))
  ggplot2::ggsave(filename = fig_name, 
                  plot = p,
                  dpi = 300,
                  width = 6, 
                  height = 4,
                  units = "in")
  p
  ggsave(fig_name, plot = p, width = 8, height = 6, dpi = 300)
}

## Moderator Analysis Plot

In [ ]:
df_mod <- read.csv("Data/ModeratorAnalysis_Synergy_Aggregate_edit.csv")
file_name <- "Figures/ModeratorAnalysis_Synergy.png"
  
# indent the subgroup if there is a number in the placebo column
  #df_mod$Subgroup <- ifelse((is.na(df_mod$estimate) | (df_mod$Subgroup == "All Effect Sizes")), 
                        #df_mod$Subgroup,
                        #paste0("   ", df_mod$Subgroup))

# NA to blank or NA will be transformed to charachter.
df_mod$n <- ifelse(is.na(df_mod$n), "", df_mod$n)

# Add three blank columns for CI
df_mod$`Human-AI system\n vs max(AI,H)` <- paste(rep(" ", 25), collapse = " ")
df_mod$`Human-AI system\n vs AI alone` <- paste(rep(" ", 25), collapse = " ")
df_mod$`Human-AI system\n vs Human alone` <- paste(rep(" ", 25), collapse = " ")

df_mod

# Set-up theme
tm <- forest_theme(base_size = 10,
                   refline_lty = "solid",
                   ci_pch = 15,
                   ci_fill = "#377eb8",
                   ci_col = "#377eb8",
                   colhead=list(fg_params=list(hjust=c(0, 0, 0.1, 0.1), x=c(.3, .3, 0.4, 0.4))),
                   # Table cell padding, width 4 and heights 3
                   core = list(padding = unit(c(4, 3), "mm")))

png(file_name, res = 300, width = 10, height = 11, units = "in")
par(mar = c(2,1,2,2))
p <- forest(df_mod[,c(1, 3, 24, 25, 26)],
            est = list(df_mod$estimate,
                       df_mod$estimate_A,
                       df_mod$estimate_H),
            lower = list(df_mod$conf.low,
                         df_mod$conf.low_A,
                         df_mod$conf.low_H), 
            upper = list(df_mod$conf.high,
                         df_mod$conf.high_A,
                         df_mod$conf.high_H),
            ci_column = c(3, 4, 5),
            ref_line = 0,
            xlim = c(-1.3, 1.3),
            arrow_lab = c("No Synergy", "Synergy"),
            # vert_line = c(0.5, 2),
            theme = tm
            )
plot(p)
dev.off()